# Modelos — STARWARS_AUTOCALLS

Entrenamiento, comparación y explicación de los modelos que predicen `avg_duration_months`.

| Bloque | Qué hace |
|---|---|
| 0 | Setup |
| 1 | Carga y contrato de features |
| 2 | Corte temporal: pasado para entrenar, futuro para evaluar |
| 3 | Referencias sin modelo (los dos baselines) |
| 4 | Cuánto aporta el plazo, y por qué predecimos el ratio |
| 5 | Modelo principal: CatBoost sobre el ratio |
| 6 | GAM explicativo |
| 7 | Comparación con intervalos de confianza |
| 8 | Backtest de origen deslizante |
| 9 | Bandas de incertidumbre P10–P90 |
| 10 | Interpretabilidad (SHAP) |
| 11 | Guardado de artefactos |

**La idea central:** un autocallable dura, como mucho, hasta su vencimiento. En vez de predecir los
meses directamente, predecimos **qué fracción de su plazo sobrevive** el producto y luego
multiplicamos por el plazo. La métrica siempre se reporta en meses, para que todo sea comparable.

## 0. Setup

Semilla fija, rutas y directorios de artefactos.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

RANDOM_SEED = 42
TARGET = "avg_duration_months"
MATURITY = "nominal_maturity_months"
HOLDOUT_DAYS = 180

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "data" / "processed").exists() else cwd.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
BASELINE_DIR = MODELS_DIR / "baseline"
GAM_DIR = MODELS_DIR / "gam"
CATBOOST_DIR = MODELS_DIR / "catboost"
REPORTS_DIR = MODELS_DIR / "reports"
for directory in [BASELINE_DIR, GAM_DIR, CATBOOST_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Proyecto  : {PROJECT_ROOT}")
print(f"Artefactos: {MODELS_DIR}")

## 1. Carga y contrato de features

El contrato lo fija `Preprocess_starwars.ipynb` y se lee de `model_feature_columns.csv`. Este
notebook **no vuelve a deducir la lista por su cuenta**: si preproceso y modelo dedujeran cada uno
sus columnas, podrían desincronizarse sin que nadie se diera cuenta.

Los asserts comprueban que lo que llega es lo que esperamos: solo RFQ ejecutadas, sin nulos, sin
texto y con el plazo dentro del contrato.

In [ ]:
data = pd.read_csv(
    PROCESSED_DIR / "train_features.csv",
    parse_dates=["requested_date", "start_date", "end_date"],
)
feature_columns = pd.read_csv(PROCESSED_DIR / "model_feature_columns.csv")["feature_name"].tolist()

assert data["executed"].all(), "train_features debe contener solo RFQs ejecutadas"
assert data[TARGET].notna().all(), "El entrenamiento requiere target observado"
assert set(feature_columns).issubset(data.columns), "El contrato pide columnas que no están"
assert data[feature_columns].select_dtypes(include="object").empty, "Las features deben ser numéricas"
assert not data[feature_columns].isna().any().any(), "Hay nulos en las features"
assert MATURITY in feature_columns, "El plazo debe formar parte del contrato"

# El ratio es la parametrizacion del target, no una feature: nunca entra en X.
data["duration_ratio"] = data[TARGET] / data[MATURITY]
assert "duration_ratio" not in feature_columns

print(f"RFQ ejecutadas : {len(data):,}")
print(f"Periodo        : {data.requested_date.min():%Y-%m-%d} a {data.requested_date.max():%Y-%m-%d}")
print(f"Features       : {len(feature_columns)}")
print()
print(f"Duración  (meses): media {data[TARGET].mean():.2f} | std {data[TARGET].std():.2f}")
print(f"Plazo     (meses): media {data[MATURITY].mean():.2f} | std {data[MATURITY].std():.2f}")
print(f"Ratio duración/plazo: media {data['duration_ratio'].mean():.3f} | std {data['duration_ratio'].std():.3f}")

## 2. Corte temporal

Entrenamos con el pasado y evaluamos con los últimos 180 días, que el modelo no ha visto nunca.
Nada de división aleatoria: en un problema con fecha, mezclar futuro y pasado infla la métrica.

In [ ]:
test_start = data["requested_date"].max().normalize() - pd.Timedelta(days=HOLDOUT_DAYS - 1)
train = data.loc[data["requested_date"] < test_start].copy()
test = data.loc[data["requested_date"] >= test_start].copy()

assert train["requested_date"].max() < test["requested_date"].min()
assert len(train) and len(test)

y_train, y_test = train[TARGET], test[TARGET]
ratio_train = train["duration_ratio"]

# Techo del ratio, calculado SOLO con train. Sirve para no dejar que el modelo
# prediga duraciones por encima de lo que se ha observado nunca.
RATIO_CAP = float(ratio_train.max())

print(pd.DataFrame({
    "split": ["train (pasado)", "test (futuro reservado)"],
    "n_rfqs": [len(train), len(test)],
    "desde": [train.requested_date.min().date(), test.requested_date.min().date()],
    "hasta": [train.requested_date.max().date(), test.requested_date.max().date()],
}).to_string(index=False))
print()
print(f"Techo del ratio observado en train: {RATIO_CAP:.3f}")

## 3. Referencias sin modelo

Un modelo solo es bueno comparado con lo que harías **sin** modelo. Usamos dos referencias:

1. **Mediana global** — "todos los productos duran lo mismo". Es el suelo absoluto.
2. **Baseline estructural** — "un producto dura una fracción fija de su plazo", es decir
   `ratio_medio × plazo`. Esto sí es lo que haría una mesa a mano, y por eso es la referencia
   honesta: batir a la mediana global es fácil, batir al baseline estructural es el trabajo real.

El baseline anterior (mediana por mes × día de la semana) se ha retirado: la duración de un
autocallable no depende del día de la semana en que se pidió el precio, así que no medía nada.

In [ ]:
# 1) Mediana global.
median_train = float(y_train.median())
pred_median = np.full(len(test), median_train)

# 2) Baseline estructural: fraccion media del plazo, calculada solo con train.
mean_ratio_train = float(ratio_train.mean())
pred_structural = mean_ratio_train * test[MATURITY].to_numpy()

print(f"Mediana de duración en train        : {median_train:.2f} meses")
print(f"Fracción media del plazo en train   : {mean_ratio_train:.3f}")
print()
print(f"MAE mediana global                  : {mean_absolute_error(y_test, pred_median):.3f} meses")
print(f"MAE baseline estructural            : {mean_absolute_error(y_test, pred_structural):.3f} meses")
print()
print(f"Lectura de negocio: sin modelo, la mesa asumiría que un producto vive el "
      f"{mean_ratio_train:.0%} de su plazo.")

## 4. Cuánto aporta el plazo, y por qué predecimos el ratio

Este bloque es la justificación de las dos decisiones de modelado. Entrenamos el **mismo** CatBoost
con la misma configuración tres veces, cambiando solo qué predice y con qué información:

| Variante | Qué ve | Qué predice |
|---|---|---|
| A — sin plazo | contrato sin `nominal_maturity_months` | meses |
| B — con plazo | contrato completo | meses |
| C — con plazo, ratio | contrato completo | fracción del plazo, y luego se multiplica |

A es el modelo tal y como estaba antes de este cambio (salvo que ya no arrastra las variables
redundantes). La diferencia A→B mide **cuánta señal aportaba la variable que faltaba**,
y la diferencia B→C mide **cuánto vale reparametrizar** el target. Las tres se miden en meses sobre
las mismas filas, así que son directamente comparables.

In [ ]:
CATBOOST_PARAMS = dict(
    loss_function="MAE", eval_metric="MAE",
    iterations=800, learning_rate=0.05, depth=8, l2_leaf_reg=5.0,
    random_seed=RANDOM_SEED, verbose=False, allow_writing_files=False,
)

def fit_catboost(X, y, **overrides):
    """Entrena CatBoost con la configuracion fija del proyecto."""
    model = CatBoostRegressor(**{**CATBOOST_PARAMS, **overrides})
    model.fit(X, y)
    return model

def ratio_to_months(model, frame, columns, cap):
    """Convierte la prediccion de ratio en meses, sin pasarse del techo observado."""
    ratio = np.clip(model.predict(frame[columns]), 0.0, cap)
    return ratio * frame[MATURITY].to_numpy()

features_sin_plazo = [c for c in feature_columns if c != MATURITY]

model_a = fit_catboost(train[features_sin_plazo], y_train)
pred_a = model_a.predict(test[features_sin_plazo])

model_b = fit_catboost(train[feature_columns], y_train)
pred_b = model_b.predict(test[feature_columns])

model_c = fit_catboost(train[feature_columns], ratio_train)
pred_c = ratio_to_months(model_c, test, feature_columns, RATIO_CAP)

ablation = pd.DataFrame([
    {"variante": "A - sin el plazo", "n_features": len(features_sin_plazo),
     "MAE_meses": mean_absolute_error(y_test, pred_a)},
    {"variante": "B - con el plazo, predice meses", "n_features": len(feature_columns),
     "MAE_meses": mean_absolute_error(y_test, pred_b)},
    {"variante": "C - con el plazo, predice ratio", "n_features": len(feature_columns),
     "MAE_meses": mean_absolute_error(y_test, pred_c)},
])
print(ablation.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print(f"Añadir el plazo        : {ablation.MAE_meses.iloc[0] - ablation.MAE_meses.iloc[1]:+.3f} meses de MAE")
print(f"Reparametrizar a ratio : {ablation.MAE_meses.iloc[1] - ablation.MAE_meses.iloc[2]:+.3f} meses de MAE")

Merece la pena entender **por qué** la reparametrización ayuda, más allá del número.

La duración cruda mezcla dos cosas: la escala del producto (su plazo) y su comportamiento (cuándo
se cancela dentro de ese plazo). Al dividir por el plazo, el modelo solo tiene que aprender lo
segundo, que es lo que realmente depende de las barreras, la volatilidad y la cesta. Es la misma
razón por la que se modelan tasas en vez de recuentos.

Y explica algo que se veía raro en el EDA: contra la duración cruda muchas variables parecían
anémicas, porque estaban contaminadas por una escala que el modelo no podía observar.

In [ ]:
# La misma variable, correlacionada contra el target crudo y contra el ratio.
comparacion = pd.DataFrame({
    "corr_con_duracion": data[feature_columns].corrwith(data[TARGET]),
    "corr_con_ratio": data[feature_columns].corrwith(data["duration_ratio"]),
})
destacadas = ["basket_size", "realized_vol_range", "protection_barrier_pct",
              "observation_frequency_months", "no_call_period_months"]
print("Correlación de cada variable con el target crudo y con el ratio:")
print(comparacion.loc[[c for c in destacadas if c in comparacion.index]].round(3).to_string())
print()
print("El sentido de negocio del efecto worst-of, medido sobre el ratio:")
print(
    data.groupby("basket_size")["duration_ratio"]
    .agg(n_rfqs="size", ratio_medio="mean").round(3)
    .rename_axis("subyacentes en la cesta")
)
print()
print("Más subyacentes -> el peor de la cesta está más abajo -> cuesta más superar la barrera")
print("-> el producto vive una fracción mayor de su plazo.")

## 5. Modelo principal: CatBoost sobre el ratio

La variante C del bloque anterior es el modelo de producción. La configuración es fija y pequeña
a propósito: sin búsqueda de hiperparámetros, porque con un holdout de este tamaño el ruido de la
métrica es mayor que casi cualquier ganancia de tuning (se ve en el bloque 7).

`loss_function="MAE"` porque la métrica de negocio es el error absoluto en meses: penalizar el
cuadrado haría que el modelo persiguiera los productos raros de duración extrema.

In [ ]:
model_primary = model_c          # ya entrenado en el bloque anterior
pred_primary = pred_c

print(f"Modelo principal: CatBoost sobre el ratio, {len(feature_columns)} features")
print(f"MAE  : {mean_absolute_error(y_test, pred_primary):.3f} meses")
print(f"RMSE : {root_mean_squared_error(y_test, pred_primary):.3f} meses")
print()
print("Predicciones frente a lo observado (primeras 8 RFQ del holdout):")
print(pd.DataFrame({
    "plazo": test[MATURITY].to_numpy(),
    "real": y_test.to_numpy(),
    "predicho": pred_primary,
    "error": y_test.to_numpy() - pred_primary,
}).head(8).round(2).to_string(index=False))

## 6. GAM explicativo

Un modelo aditivo con splines para las variables continuas y términos lineales para las dummies.
No compite con CatBoost: existe para poder **leer** los efectos uno a uno, sin interacciones.

Dos detalles frente a la versión anterior: ahora predice el ratio (para ser comparable con el
modelo principal) y las columnas lineales van estandarizadas. Sin estandarizar, la penalización
de Ridge caía de forma desigual sobre variables en escalas distintas y los coeficientes no eran
comparables entre sí — que es exactamente para lo que se usaban.

In [ ]:
gam_spline_columns = [
    "autocall_barrier_pct", "protection_barrier_pct", "no_call_period_months",
    "observation_frequency_months", "quoted_implied_vol", "log_notional_credits",
    "basket_size", "realized_vol_mean", "realized_vol_range",
    "structural_base_vol_mean", "structural_base_vol_range",
    "requested_month_sin", "requested_month_cos", "requested_dayofweek",
    MATURITY,
]
gam_spline_columns = [c for c in gam_spline_columns if c in feature_columns]
gam_linear_columns = [c for c in feature_columns if c not in gam_spline_columns]

gam = Pipeline([
    ("features", ColumnTransformer([
        ("spline", SplineTransformer(n_knots=5, degree=3, extrapolation="linear"), gam_spline_columns),
        ("linear", StandardScaler(), gam_linear_columns),
    ], remainder="drop")),
    ("regressor", Ridge(alpha=10.0)),
])
gam.fit(train[feature_columns], ratio_train)
pred_gam = ratio_to_months(gam, test, feature_columns, RATIO_CAP)

print(f"Splines: {len(gam_spline_columns)} | Términos lineales: {len(gam_linear_columns)}")
print(f"MAE del GAM: {mean_absolute_error(y_test, pred_gam):.3f} meses")

## 7. Comparación con intervalos de confianza

Una tabla de MAE sin intervalo invita a leer diferencias que no existen. Con 746 filas de holdout,
el error estándar del MAE ronda el medio mes: **dos modelos que se llevan 0.2 meses son
indistinguibles**, y ordenarlos como si uno fuese mejor es sobre-interpretar el ruido.

El intervalo se calcula por bootstrap: se remuestrean las filas del holdout 2.000 veces y se mira
cómo se mueve el MAE.

In [ ]:
def bootstrap_mae_ci(y_true, y_pred, n_samples=2000, seed=RANDOM_SEED):
    """Intervalo de confianza al 95% del MAE, remuestreando las filas del holdout."""
    rng = np.random.default_rng(seed)
    errors = np.abs(np.asarray(y_true, dtype=float) - np.asarray(y_pred, dtype=float))
    draws = errors[rng.integers(0, len(errors), size=(n_samples, len(errors)))].mean(axis=1)
    return float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))

candidatos = {
    "Mediana global": pred_median,
    "Baseline estructural (ratio x plazo)": pred_structural,
    "CatBoost sin el plazo": pred_a,
    "GAM sobre el ratio": pred_gam,
    "CatBoost sobre el ratio (principal)": pred_primary,
}

filas = []
for nombre, pred in candidatos.items():
    low, high = bootstrap_mae_ci(y_test, pred)
    filas.append({
        "modelo": nombre,
        "MAE_meses": mean_absolute_error(y_test, pred),
        "IC95_bajo": low,
        "IC95_alto": high,
        "RMSE_meses": root_mean_squared_error(y_test, pred),
    })
metrics = pd.DataFrame(filas).sort_values("MAE_meses").reset_index(drop=True)
metrics.to_csv(REPORTS_DIR / "holdout_metrics.csv", index=False)
print(metrics.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
mejora = mean_absolute_error(y_test, pred_structural) - mean_absolute_error(y_test, pred_primary)
print(f"El modelo principal mejora al baseline estructural en {mejora:.2f} meses de MAE.")

## 8. Backtest de origen deslizante

Un único corte temporal mide un semestre concreto, y ese semestre puede ser fácil o difícil.
Repetimos el ejercicio sobre **cinco ventanas consecutivas de seis meses**, entrenando cada vez
solo con lo anterior a la ventana. Así se ve si la ventaja del plazo es estable o fue suerte de un
periodo.

Es la traducción literal del `build_tag` del proyecto: *fenêtre glissante*, ventana deslizante.

In [ ]:
BACKTEST_FOLDS = 5
FOLD_MONTHS = 6

ultima_fecha = data["requested_date"].max().normalize()
bordes = [ultima_fecha - pd.DateOffset(months=FOLD_MONTHS * i) for i in range(BACKTEST_FOLDS, -1, -1)]

filas = []
for i in range(BACKTEST_FOLDS):
    inicio, fin = bordes[i], bordes[i + 1]
    fold_train = data.loc[data["requested_date"] <= inicio]
    fold_test = data.loc[(data["requested_date"] > inicio) & (data["requested_date"] <= fin)]
    if len(fold_test) < 50:
        continue

    fold_cap = float(fold_train["duration_ratio"].max())
    fold_y = fold_test[TARGET]

    sin_plazo = fit_catboost(fold_train[features_sin_plazo], fold_train[TARGET])
    con_ratio = fit_catboost(fold_train[feature_columns], fold_train["duration_ratio"])

    filas.append({
        "ventana": f"{inicio.date()} -> {fin.date()}",
        "n_train": len(fold_train),
        "n_test": len(fold_test),
        "MAE_estructural": mean_absolute_error(
            fold_y, fold_train["duration_ratio"].mean() * fold_test[MATURITY].to_numpy()),
        "MAE_sin_plazo": mean_absolute_error(fold_y, sin_plazo.predict(fold_test[features_sin_plazo])),
        "MAE_principal": mean_absolute_error(
            fold_y, ratio_to_months(con_ratio, fold_test, feature_columns, fold_cap)),
    })

backtest = pd.DataFrame(filas)
backtest.to_csv(REPORTS_DIR / "backtest_metrics.csv", index=False)
print(backtest.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print("Media sobre las ventanas:")
for col in ["MAE_estructural", "MAE_sin_plazo", "MAE_principal"]:
    print(f"  {col:<18} {backtest[col].mean():6.3f} meses  (desviación entre ventanas {backtest[col].std():.3f})")
print()
gana = (backtest["MAE_principal"] < backtest["MAE_sin_plazo"]).sum()
print(f"El modelo con plazo gana en {gana} de {len(backtest)} ventanas.")

## 9. Bandas de incertidumbre P10–P90

Una mesa de riesgo no gestiona un número, gestiona un rango: *"este producto vive entre 14 y 38
meses"* es más accionable que *"vive 24 meses"*. Entrenamos dos CatBoost más con pérdida de
cuantil sobre el mismo ratio, para los percentiles 10 y 90.

La comprobación importante es la **cobertura**: si las bandas están bien calibradas, alrededor del
80% de las duraciones reales deberían caer dentro. Si sale muy por debajo, las bandas son
demasiado estrechas y darían una falsa sensación de precisión.

In [ ]:
model_p10 = fit_catboost(train[feature_columns], ratio_train, loss_function="Quantile:alpha=0.1",
                         eval_metric="Quantile:alpha=0.1")
model_p90 = fit_catboost(train[feature_columns], ratio_train, loss_function="Quantile:alpha=0.9",
                         eval_metric="Quantile:alpha=0.9")

banda_baja = ratio_to_months(model_p10, test, feature_columns, RATIO_CAP)
banda_alta = ratio_to_months(model_p90, test, feature_columns, RATIO_CAP)
# Son dos modelos independientes: nada les impide cruzarse en alguna fila.
banda_baja, banda_alta = np.minimum(banda_baja, banda_alta), np.maximum(banda_baja, banda_alta)

dentro = ((y_test.to_numpy() >= banda_baja) & (y_test.to_numpy() <= banda_alta)).mean()
print(f"Cobertura observada de la banda P10-P90 : {dentro:.1%}  (objetivo 80%)")
print(f"Anchura media de la banda               : {(banda_alta - banda_baja).mean():.2f} meses")
print()
print("Ejemplo de salida para la mesa (primeras 8 RFQ del holdout):")
print(pd.DataFrame({
    "plazo": test[MATURITY].to_numpy(),
    "P10": banda_baja,
    "estimación": pred_primary,
    "P90": banda_alta,
    "real": y_test.to_numpy(),
}).head(8).round(1).to_string(index=False))

## 10. Interpretabilidad

SHAP mide cuánto mueve cada variable la predicción, no el efecto causal de tocarla: describe lo que
el modelo usa, no lo que pasaría si la mesa cambiara una barrera.

Como el modelo predice el ratio, los valores SHAP salen en unidades de ratio. La columna de meses
es una conversión aproximada (multiplicando por el plazo medio del holdout) para poder leerlos con
intuición de negocio.

In [ ]:
shap_values = model_primary.get_feature_importance(type="ShapValues", data=Pool(test[feature_columns]))
plazo_medio = float(test[MATURITY].mean())
shap_importance = (
    pd.DataFrame({
        "feature": feature_columns,
        "shap_medio_abs_ratio": np.abs(shap_values[:, :-1]).mean(axis=0),
    })
    .assign(aprox_meses=lambda d: d["shap_medio_abs_ratio"] * plazo_medio)
    .sort_values("shap_medio_abs_ratio", ascending=False)
    .reset_index(drop=True)
)
shap_importance.to_csv(REPORTS_DIR / "catboost_shap_importance.csv", index=False)
print(shap_importance.head(15).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

grafico = shap_importance.head(12).sort_values("shap_medio_abs_ratio")
ax = grafico.plot.barh(x="feature", y="aprox_meses", legend=False, figsize=(8, 5))
ax.set(xlabel="Impacto medio en la predicción (meses aprox.)", ylabel="",
       title="CatBoost: qué variables mueven la duración estimada")
plt.tight_layout()
plt.show()

# Coeficientes del GAM, ya comparables entre si porque las columnas van estandarizadas.
gam_terms = gam.named_steps["features"].get_feature_names_out()
gam_linear_effects = (
    pd.DataFrame({"term": gam_terms, "coefficient": gam.named_steps["regressor"].coef_})
    .loc[lambda d: d["term"].str.startswith("linear__")]
    .assign(abs_coefficient=lambda d: d["coefficient"].abs())
    .sort_values("abs_coefficient", ascending=False)
)
gam_linear_effects.to_csv(REPORTS_DIR / "gam_linear_effects.csv", index=False)
print()
print("GAM - efectos lineales más fuertes (en unidades de ratio por desviación típica):")
print(gam_linear_effects.head(10).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

## 11. Guardado de artefactos

El contrato JSON es lo que hace reproducible la inferencia: guarda las columnas, el orden, el techo
del ratio y **cómo convertir la salida del modelo en meses**. La API lo lee y no necesita saber
nada más de este notebook.

In [ ]:
model_primary.save_model(str(CATBOOST_DIR / "model.cbm"))
model_p10.save_model(str(CATBOOST_DIR / "model_p10.cbm"))
model_p90.save_model(str(CATBOOST_DIR / "model_p90.cbm"))
joblib.dump(gam, GAM_DIR / "model.joblib")

feature_contract = {
    "target": TARGET,
    "unit": "months",
    # El modelo predice la fraccion del plazo; la API multiplica por nominal_maturity_months.
    "target_parametrization": "duration_ratio",
    "maturity_column": MATURITY,
    "maturity_range_months": [float(train[MATURITY].min()), float(train[MATURITY].max())],
    "ratio_cap": RATIO_CAP,
    "feature_columns": feature_columns,
    "time_column": "requested_date",
    "holdout_start": str(test_start.date()),
    "primary_model": "model.cbm",
    "quantile_models": {"p10": "model_p10.cbm", "p90": "model_p90.cbm"},
}
(CATBOOST_DIR / "feature_contract.json").write_text(
    json.dumps(feature_contract, indent=2, ensure_ascii=False), encoding="utf-8"
)
(BASELINE_DIR / "structural_baseline.json").write_text(
    json.dumps({"mean_duration_ratio": mean_ratio_train,
                "median_duration_months": median_train}, indent=2), encoding="utf-8"
)

resumen = f"""# Resultados de modelos

Holdout temporal: {test.requested_date.min():%Y-%m-%d} a {test.requested_date.max():%Y-%m-%d} ({len(test):,} RFQs).
Modelo principal: CatBoost sobre el ratio duración/plazo, {len(feature_columns)} variables.

| Modelo | MAE (meses) | IC95% | RMSE (meses) |
|---|---:|:--:|---:|
"""
resumen += "\n".join(
    f"| {r.modelo} | {r.MAE_meses:.3f} | {r.IC95_bajo:.2f} – {r.IC95_alto:.2f} | {r.RMSE_meses:.3f} |"
    for r in metrics.itertuples(index=False)
)
resumen += f"""

## Backtest de origen deslizante ({len(backtest)} ventanas de {FOLD_MONTHS} meses)

| Variante | MAE medio | Desviación entre ventanas |
|---|---:|---:|
| Baseline estructural | {backtest.MAE_estructural.mean():.3f} | {backtest.MAE_estructural.std():.3f} |
| CatBoost sin plazo | {backtest.MAE_sin_plazo.mean():.3f} | {backtest.MAE_sin_plazo.std():.3f} |
| CatBoost sobre el ratio | {backtest.MAE_principal.mean():.3f} | {backtest.MAE_principal.std():.3f} |

## Decisiones

- El plazo del producto (`nominal_maturity_months`) entra en el contrato: es un término pactado
  de la RFQ, no información del futuro. Aporta {ablation.MAE_meses.iloc[0] - ablation.MAE_meses.iloc[1]:.2f} meses de MAE.
- El modelo predice la fracción del plazo que sobrevive el producto, no los meses directamente.
  Aporta otros {ablation.MAE_meses.iloc[1] - ablation.MAE_meses.iloc[2]:.2f} meses.
- Las diferencias por debajo de ~0.5 meses no son distinguibles del ruido con este holdout.
- Se publican bandas P10-P90 con cobertura observada del {dentro:.0%}.
"""
(MODELS_DIR / "README.md").write_text(resumen, encoding="utf-8")

print("Artefactos guardados:")
for path in sorted(MODELS_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(MODELS_DIR)}")

## Controles de fuga de información

- El plazo entra de forma consciente y justificada (bloque 3 del preproceso). Todo lo demás que se
  conoce después de cotizar —target, ejecución, fechas— queda fuera.
- La volatilidad de mercado es la última publicada **estrictamente antes** de la RFQ.
- El techo del ratio, la mediana y la fracción media se calculan **solo con train**.
- Todos los modelos se entrenan solo con el bloque pasado, y en el backtest cada ventana reentrena
  desde cero con lo anterior a ella.

## Limitaciones

- **Sesgo de selección**: solo hay target para las RFQ ejecutadas (55% del total). El modelo se
  aplica a todas al cotizar, así que asume que las no ejecutadas se comportan igual. Corregirlo
  formalmente pediría un modelo de selección tipo Heckman o ponderación por propensión.
- **El plazo debe venir en la petición.** Para una RFQ sin plazo definido el modelo principal no
  aplica; la variante A del bloque 4 queda como respaldo documentado.
- La volatilidad implícita cotizada aporta muy poca señal, porque es casi una función determinista
  de la volatilidad estructural, que ya está en el modelo.
- SHAP y los coeficientes del GAM describen asociaciones predictivas, no relaciones causales.